In [1]:
import numpy as np
import pandas as pd
pd.set_option("display.float_format", "{:.3f}".format)

from scipy.stats import norm
from scipy.linalg import eig

import statsmodels.api as sm
import statsmodels.formula.api as smf

import matplotlib.pyplot as plt

# Reproducibility
np.random.seed(53241986)

## Synthetic Data Generation and Inference Workflow

In this notebook, we generate synthetic data from an individual-based simulation (IBM) of a species’ population cycle, using fixed true parameters for the vital-rate models. We assume a time-homogeneous demographic process, meaning that the kernel function and vital-rate functions do not depend on time.

This simulation framework follows the IBM approach used for monocarp species in Ellner and Rees (2016), based on the code here:
https://github.com/ipmbook/first-edition/blob/master/Rcode/c2/Monocarp%20Calculations.R

### Vital rates

We model five vital rates:

1. Growth(z1, z) ~ Normal(f(size), sigma)
2. Flowering ~ Bernoulli(sigmoid(f(size)))
3. Survival ~ Bernoulli(sigmoid(f(size)))
4. Seed production ~ Poisson(exp(f(size)))
5. Recruit size ~ N(f(size), sigma)

### Simulation procedure

* We simulate the IBM for 100 years, starting from an initial population of 500 plants.
* The first 10 years are discarded, since the initial population is arbitrary and has not yet reached the stable population structure generated by the life cycle process.
* Each row of the resulting dataset represents one individual observed over one time interval.
* We then subsample the population to mimic random sampling of individuals.

This subsampling is valid because the demographic process is assumed to be time invariant. Under this assumption, observing an individual over years 4–5 is equivalent to observing another individual over years 67–68.

### Inference

Using the subsampled data as if it were real observations, we infer the true model parameters.

The vital-rate functions are fitted using generalized linear models (GLMs).

### IPM construction

Using both the true parameters and the estimated parameters, we construct integral projection models (IPMs). The integral is discretized using the midpoint method, which converts the kernel function into a kernel matrix whose entries represent transition probabilities. These probabilities are obtained from the fitted model parameters.

Finally, we compute population-level statistics for both the true and estimated kernel matrices and compare the results.


In [2]:
# True model parameters
m_par_true = {
    "surv.int": -0.65, #intercept
    "surv.z": 0.75,    #slope

    "flow.int": -18.0,
    "flow.z": 6.9,

    "grow.int": 0.96,
    "grow.z": 0.59,
    "grow.sd": 0.67,

    "rcsz.int": -0.08,
    "rcsz.sd": 0.76,

    "seed.int": 1.00,
    "seed.z": 2.20,

    "p.r": 0.007
}

In [3]:
# Growth kernel
def G_z1z(z1, z, m_par):
    """
    Growth probability density

    args:
    zi: float or numpy array [size at time t+1]
    z: float or numpy array [size at time t]
    m_par: dict [dictionary containing demographic parameters]

    returns:
    float or numpy array [probability density of growing from z to z1]
    """
    mean_size_next_year = (
        m_par["grow.int"] + m_par["grow.z"] * z
    )

    growth_sd = m_par["grow.sd"]

    probability_density = norm.pdf(
        z1,
        loc=mean_size_next_year,
        scale=growth_sd
    )

    return probability_density


In [4]:
def sigmoid(x):
    return 1/(1 + np.exp(-x))
    
# Survival Kernel
def s_z(z, m_par):
    """
    Survival Probability as a Function of Size
    """

    linear_predictor = (
        m_par["surv.int"] + m_par["surv.z"] * z
    )

    survival_probability = sigmoid(linear_predictor)
    return survival_probability

In [5]:
# Flowering Probability
def p_bz(z, m_par):
    """
    Probability that a plant flowers.
    """

    linear_predictor = (
        m_par["flow.int"] +
        m_par["flow.z"] * z
    )

    flowering_probability = sigmoid(linear_predictor)

    return flowering_probability

In [6]:
# Seed Production [Poisson mean function]
def b_z(z, m_par):
    """
    Expected number of seeds produced by a flowering plant.
    """
    linear_predictor = m_par["seed.int"] + m_par["seed.z"] * z

    expected_seed_number = np.exp(linear_predictor)

    return expected_seed_number

In [7]:
# Probability density of recruit size
def c_0z1(z1, m_par):
    """
    Probability density of recruit size.
    """

    recruit_mean_size = m_par["rcsz.int"]

    recruit_standard_deviation = m_par["rcsz.sd"]

    recruit_density = norm.pdf(
        z1,
        loc=recruit_mean_size,
        scale=recruit_standard_deviation
    )

    return recruit_density

### IBM Simulations


In [50]:
# NOTE: Increase initial population size from 250 to 500 to prevent extinction 
# from demographic stochasticity
n_years = 101
init_pop_size = 500 
rng = np.random.default_rng(10)

plant_sizes = rng.normal(
    loc = m_par_true['rcsz.int'],
    scale = m_par_true['rcsz.sd'],
    size = init_pop_size
 )

plant_ages = np.zeros(init_pop_size, dtype = int)
mean_age_history = [plant_ages.mean()]
simulation_records = []
recruits_per_year = []
year = 1
while (year != n_years) and (len(plant_sizes) < 1_500_000):
#for year in range(1, n_years + 1):
    current_population_size = len(plant_sizes)
    print(f"Year {year}:Current Population Size = {current_population_size}")

    flowered = rng.binomial(
        n = 1,
        p = p_bz(plant_sizes, m_par_true)
    )

    number_flowering = flowered.sum()
    seed_counts = np.full(
        shape = len(plant_sizes),
        fill_value = np.nan
    )

    seed_counts[flowered == 1] = rng.poisson(
        lam = b_z(
            plant_sizes[flowered == 1],
            m_par_true
        ) 
    )

    total_seeds = np.nansum(seed_counts)

    if number_flowering == 0:
        number_recruits = 0
    else:
        # Each seed has some probability of becoming a new recruit
        number_recruits = rng.binomial(
            n = int(total_seeds),
            p = m_par_true["p.r"]
        )
    recruits_per_year.append(number_recruits)

    recruit_sizes = rng.normal(
        loc = m_par_true["rcsz.int"],
        scale = m_par_true["rcsz.sd"],
        size = number_recruits
    )

    survived = np.full(
        shape = len(plant_sizes),
        fill_value = np.nan
        )

    non_flowering = (flowered == 0)
    survival_probabilities = s_z(
        plant_sizes[non_flowering],
        m_par_true
    )

    survived[non_flowering] = rng.binomial(
        n = 1,
        p = survival_probabilities
    )

    still_alive = ((flowered == 0) & (survived ==1))
    expected_next_size = (
        m_par_true["grow.int"] + m_par_true["grow.z"] * plant_sizes[still_alive]
    )

    grown_sizes = rng.normal(
        loc = expected_next_size,
        scale = m_par_true["grow.sd"]
    )

    next_year_size = np.full(len(plant_sizes), np.nan)
    next_year_size[still_alive] = grown_sizes

    for i in range(len(plant_sizes)):
        simulation_records.append({
            "z": plant_sizes[i],
            "Repr": flowered[i],
            "Seeds": seed_counts[i],
            "Surv": survived[i],
            "z1": next_year_size[i],
            "age": plant_ages[i],
            "alive": still_alive[i],
            "yr": year
        })

    plant_sizes = np.concatenate([recruit_sizes, grown_sizes])

    plant_ages = np.concatenate([
        np.zeros(number_recruits, dtype=int),
        plant_ages[still_alive] + 1
    ])
    year += 1

sim_data = pd.DataFrame(simulation_records)

Year 1:Current Population Size = 500
Year 2:Current Population Size = 183
Year 3:Current Population Size = 152
Year 4:Current Population Size = 259
Year 5:Current Population Size = 312
Year 6:Current Population Size = 226
Year 7:Current Population Size = 335
Year 8:Current Population Size = 231
Year 9:Current Population Size = 259
Year 10:Current Population Size = 290
Year 11:Current Population Size = 305
Year 12:Current Population Size = 228
Year 13:Current Population Size = 210
Year 14:Current Population Size = 283
Year 15:Current Population Size = 496
Year 16:Current Population Size = 407
Year 17:Current Population Size = 307
Year 18:Current Population Size = 301
Year 19:Current Population Size = 346
Year 20:Current Population Size = 300
Year 21:Current Population Size = 314
Year 22:Current Population Size = 406
Year 23:Current Population Size = 358
Year 24:Current Population Size = 390
Year 25:Current Population Size = 252
Year 26:Current Population Size = 354
Year 27:Current Popul

In [51]:
sim_data.describe()

,z,Repr,Seeds,Surv,z1,age,yr
count,352846.000,352846.000,13263.000,339583.000,143210.000,352846.000,352846.000
mean,0.494,0.038,2519.246,0.422,1.422,0.741,82.407
std,1.088,0.190,5828.236,0.494,0.874,1.255,18.460
min,-3.651,0.000,37.000,0.000,-2.386,0.000,1.000
25%,-0.295,0.000,843.000,0.000,0.824,0.000,75.000
50%,0.396,0.000,1408.000,0.000,1.430,0.000,89.000
75%,1.234,0.000,2538.000,1.000,2.022,1.000,96.000
max,5.482,1.000,469842.000,1.000,5.482,19.000,100.000


In [52]:
sim_data = sim_data[sim_data["yr"] > 10].copy()

sim_data["yr"] = sim_data["yr"] - 10

In [56]:
dataset_sizes = [250] #,500,750,1000,2000]
dataset_dict = {}

for n in dataset_sizes:
    data_sample = sim_data.sample(
    n=n,
    replace=False,
    random_state=53241986
    )
    data_sample["z_classes"] = pd.cut(
        sim_data["z"],
        bins = 20
    )
    dataset_dict[f"sim_data{n}"] = data_sample
    data_sample.to_csv(f"sim_data{n}.csv", index=False)

In [57]:
sim_data = dataset_dict["sim_data250"]

In [58]:
sim_data.head()

,z,Repr,Seeds,Surv,z1,age,alive,yr,z_classes
30297,0.511,0,NaN,1.000,1.447,0,True,45,"(0.459, 0.915]"
78721,1.770,0,NaN,1.000,1.622,3,True,62,"(1.372, 1.829]"
153014,0.588,0,NaN,1.000,2.586,0,True,76,"(0.459, 0.915]"
145700,-0.109,0,NaN,0.000,NaN,0,False,75,"(-0.455, 0.00206]"
299505,0.121,0,NaN,0.000,NaN,0,False,88,"(0.00206, 0.459]"


In [59]:
growth_data = sim_data.dropna(subset = ["z1"]).copy()

In [60]:
print(len(growth_data))

93


In [61]:
growth_model = smf.ols(
    formula="z1 ~ z",
    data=growth_data
).fit()

print(growth_model.summary())

                            OLS Regression Results                            
Dep. Variable:                     z1   R-squared:                       0.332
Model:                            OLS   Adj. R-squared:                  0.325
Method:                 Least Squares   F-statistic:                     45.29
Date:                Wed, 15 Jul 2026   Prob (F-statistic):           1.48e-09
Time:                        12:18:13   Log-Likelihood:                -87.470
No. Observations:                  93   AIC:                             178.9
Df Residuals:                      91   BIC:                             184.0
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.9624      0.090     10.736      0.0

In [62]:
growth_intercept = growth_model.params["Intercept"]
growth_slope = growth_model.params["z"]
growth_sd = np.sqrt(growth_model.scale)

In [63]:
flowering_model = smf.glm(
    formula="Repr ~ z",
    data=sim_data,
    family=sm.families.Binomial()
).fit()
flowering_intercept = flowering_model.params["Intercept"]
flowering_slope = flowering_model.params["z"]
print(flowering_model.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                   Repr   No. Observations:                  250
Model:                            GLM   Df Residuals:                      248
Model Family:                Binomial   Df Model:                            1
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -5.2666
Date:                Wed, 15 Jul 2026   Deviance:                       10.533
Time:                        12:18:16   Pearson chi2:                     29.6
No. Iterations:                    11   Pseudo R-squ. (CS):             0.1427
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    -18.6783      6.197     -3.014      0.0

In [64]:
# Keep only non-flowering plants
survival_data = sim_data[sim_data["Repr"] == 0].copy()

# Fit logistic regression
survival_model = smf.glm(
    formula="Surv ~ z",
    data=survival_data,
    family=sm.families.Binomial()
).fit()

print(survival_model.summary())

# Extract coefficients
survival_intercept = survival_model.params["Intercept"]
survival_slope = survival_model.params["z"]

                 Generalized Linear Model Regression Results                  
Dep. Variable:                   Surv   No. Observations:                  245
Model:                            GLM   Df Residuals:                      243
Model Family:                Binomial   Df Model:                            1
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -145.30
Date:                Wed, 15 Jul 2026   Deviance:                       290.60
Time:                        12:18:17   Pearson chi2:                     241.
No. Iterations:                     4   Pseudo R-squ. (CS):             0.1320
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -0.8950      0.165     -5.419      0.0

In [65]:
# Keep only flowering plants
seed_data = sim_data[sim_data["Repr"] == 1].copy()

# Fit Poisson regression
seed_model = smf.glm(
    formula="Seeds ~ z",
    data=seed_data,
    family=sm.families.Poisson()
).fit()

print(seed_model.summary())

# Extract coefficients
seed_intercept = seed_model.params["Intercept"]
seed_slope = seed_model.params["z"]

                 Generalized Linear Model Regression Results                  
Dep. Variable:                  Seeds   No. Observations:                    5
Model:                            GLM   Df Residuals:                        3
Model Family:                 Poisson   Df Model:                            1
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -23.884
Date:                Wed, 15 Jul 2026   Deviance:                       3.3522
Time:                        12:18:18   Pearson chi2:                     3.33
No. Iterations:                     5   Pseudo R-squ. (CS):              1.000
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.8074      0.218      3.706      0.0

In [66]:
# Recruits have age 0
recruit_data = sim_data[sim_data["age"] == 0].copy()

recruit_mean = recruit_data["z"].mean()
recruit_sd = recruit_data["z"].std()

print(recruit_mean)
print(recruit_sd)

-0.17023136208072395
0.676555491216711


In [67]:
sim_data

,z,Repr,Seeds,Surv,z1,age,alive,yr,z_classes
30297,0.511,0,NaN,1.000,1.447,0,True,45,"(0.459, 0.915]"
78721,1.770,0,NaN,1.000,1.622,3,True,62,"(1.372, 1.829]"
153014,0.588,0,NaN,1.000,2.586,0,True,76,"(0.459, 0.915]"
145700,-0.109,0,NaN,0.000,NaN,0,False,75,"(-0.455, 0.00206]"
299505,0.121,0,NaN,0.000,NaN,0,False,88,"(0.00206, 0.459]"
...,...,...,...,...,...,...,...,...,...
229128,0.654,0,NaN,0.000,NaN,1,False,83,"(0.459, 0.915]"
257189,1.470,0,NaN,0.000,NaN,3,False,85,"(1.372, 1.829]"
334668,-0.408,0,NaN,1.000,1.141,0,True,90,"(-0.455, 0.00206]"
330854,2.004,0,NaN,1.000,2.675,5,True,89,"(1.829, 2.285]"


In [68]:
p_r_est = (
    (sim_data["age"] == 0).sum()
    /
    np.nansum(sim_data["Seeds"])
)

print(p_r_est)

0.021564913303220166


In [69]:
m_par_est = {
    "surv.int": survival_intercept,
    "surv.z": survival_slope,

    "flow.int": flowering_intercept,
    "flow.z": flowering_slope,

    "grow.int": growth_intercept,
    "grow.z": growth_slope,
    "grow.sd": growth_sd,

    "rcsz.int": recruit_mean,
    "rcsz.sd": recruit_sd,

    "seed.int": seed_intercept,
    "seed.z": seed_slope,

    "p.r": p_r_est
}

In [70]:
def P_z1z(z1, z, m_par):
    """
    Survival-growth kernel.
    """

    return (
        (1 - p_bz(z, m_par))
        * s_z(z, m_par)
        * G_z1z(z1, z, m_par)
    )

In [71]:
def F_z1z(z1, z, m_par):
    """
    Fecundity kernel.
    """

    return (
        p_bz(z, m_par)
        * b_z(z, m_par)
        * m_par["p.r"]
        * c_0z1(z1, m_par)
    )

In [72]:
def mk_K(n_mesh_points, m_par, lower_size, upper_size):

    # Width of each mesh interval
    mesh_width = (upper_size - lower_size) / n_mesh_points

    # Midpoints of each interval
    mesh_points = (
        lower_size
        + (np.arange(n_mesh_points) + 0.5)
        * mesh_width
    )

    # Evaluate kernels
    z1_grid, z_grid = np.meshgrid(
        mesh_points,
        mesh_points,
        indexing="ij"
    )

    P = (           # P is a (n_mesh_points x n_mesh_points) matrix of transition probabilities
        mesh_width
        * P_z1z(z1_grid, z_grid, m_par)
    )

    F = (           # P is a (n_mesh_points x n_mesh_points) matrix of fecundity probabilities
        mesh_width
        * F_z1z(z1_grid, z_grid, m_par)
    )

    K = P + F

    return {
        "K": K,
        "P": P,
        "F": F,
        "mesh_points": mesh_points
    }

In [73]:
n_big_matrix = 250

IPM_true = mk_K(
    n_mesh_points=n_big_matrix,
    m_par=m_par_true,
    lower_size=-2.65,
    upper_size=4.5
)

IPM_est = mk_K(
    n_mesh_points=n_big_matrix,
    m_par=m_par_est,
    lower_size=-2.65,
    upper_size=4.5
)

In [74]:
eigenvalues_true, eigenvectors_true = np.linalg.eig(IPM_true["K"])

lambda_true = np.max(eigenvalues_true.real)

print(lambda_true)

1.0593068095360045


In [75]:
eigenvalues_est, eigenvectors_est = np.linalg.eig(IPM_est["K"])

lambda_est = np.max(eigenvalues_est.real)

print(lambda_est)

1.073051763319928


In [76]:
dominant_index = np.argmax(eigenvalues_est.real)

stable_distribution = (
    eigenvectors_est[:, dominant_index].real
)

stable_distribution /= stable_distribution.sum()

In [77]:
mean_size = np.sum(
    stable_distribution
    * IPM_est["mesh_points"]
)

print(mean_size)

0.31387952205196235
